In [2]:
import enipy3 as lx
import numpy as np
import uproot
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from iminuit import Minuit
from sympy import var, solve
import math 
%matplotlib inline 

#####Vectorial rotation function 
def rotve(v,erot,angle):
    rotmeasure=np.linalg.norm(erot)
    erot=erot/rotmeasure;
    norme=np.dot(v,erot)
    vplane=v-norme*erot
    plnorm=np.linalg.norm(vplane)
    ep=vplane/plnorm
    eo=np.cross(erot,ep)
    vrot=(np.cos(angle)*ep+np.sin(angle)*eo)*plnorm+norme*erot
    return(vrot)

#####Simple asymmetric gaussian function
def gauss(x,a,b,c,p):
    p = 0
    z = (x-b)/c
    return a*np.exp(-0.5*z**2)*(1 + p*z)

#####Double asymmetric gaussian function
def gauss_2(x,a1,b1,c1,p1,a2,b2,c2,p2):
    p1 = 0
    p2 = 0
    z1 = (x-b1)/c1
    z2 = (x-b2)/c2
    return a1*np.exp(-0.5*z1**2)*(1 + p1*z1) + a2*np.exp(-0.5*z2**2)*(1 + p2*z2)

#####Triple asymmetric gaussian function
def gauss_3(x,a1,b1,c1,p1,a2,b2,c2,p2,a3,b3,c3,p3): 
    p1 = 0
    p2 = 0
    p3 = 0
    z1 = (x-b1)/c1
    z2 = (x-b2)/c2
    z3 = (x-b3)/c3
    return a1*np.exp(-0.5*z1**2)*(1 + p1*z1) + a2*np.exp(-0.5*z2**2)*(1 + p2*z2) + a3*np.exp(-0.5*z3**2)*(1 + p3*z3)

#####Minuit function for simple gaussian 
def chi1(a,b,c,p):
    return sum((gauss(x,a,b,c,p) - y)**2 for x, y in zip(pulsex, pulsey))

#####Minuit function for double gaussian 
def chi2(a1,b1,c1,p1,a2,b2,c2,p2):
    return sum((gauss_2(x,a1,b1,c1,p1,a2,b2,c2,p2) - y)**2 for x, y in zip(pulsex, pulsey))

#####Minuit function for triple gaussian 
def chi3(a1,b1,c1,p1,a2,b2,c2,p2,a3,b3,c3,p3):
    return sum((gauss_3(x,a1,b1,c1,p1,a2,b2,c2,p2,a3,b3,c3,p3) - y)**2 for x, y in zip(pulsex, pulsey))
    
######Parameters from simple gaussian fit
def get1(amp, cen, amp_err):
    sigma1 = 1
    pp = 0         
    #####Simple gaussian fit optimiser       
    m = Minuit(chi1, a=amp, b=cen, c=sigma1, p=pp, error_a=amp_err, errordef=1., pedantic=False)
    m.migrad() # run optimiser   
    ##chi square/dof
    chi_s = m.fval / (len(pulsex) - len(m.args))
    
    return np.array([m.values[0], m.values[1], m.values[2], m.values[3], 
                     m.errors[0], m.errors[1], m.errors[2], m.errors[3],chi_s])

######Parameters from double gaussian fit
def get2(amp1, cen1, amp2, cen2, amp1_err, amp2_err):
    sigma1 = 1
    sigma2 = 1
    pp1 = 0
    pp2 = 0          
    #####Double gaussian fit optimiser     
    m = Minuit(chi2, a1=amp1, b1=cen1, c1=sigma1, p1=pp1, a2=amp2, b2=cen2, c2=sigma2, p2=pp2, 
               error_a1=amp1_err, error_a2=amp2_err, errordef=1., pedantic=False)
    m.migrad() # run optimiser
    ##chi square/dof
    chi_s = m.fval / (len(pulsex) - len(m.args)) 
    return np.array([m.values[0], m.values[1], m.values[2],m.values[3], 
                     m.values[4], m.values[5], m.values[6], m.values[7],
                     m.errors[0], m.errors[1], m.errors[2], m.errors[3],
                     m.errors[4], m.errors[5], m.errors[6], m.errors[7],
                     chi_s])

######Parameters from triple gaussian fit
def get3(amp1, cen1, amp2, cen2, amp3, cen3, amp1_err, amp2_err, amp3_err):
    sigma1 = 1
    sigma2 = 1
    sigma3 = 1
    pp1 = 0
    pp2 = 0
    pp3 = 0
    #####Triple gaussian fit optimiser     
    m = Minuit(chi3, a1=amp1, b1=cen1, c1=sigma1, p1=pp1, a2=amp2, b2=cen2, c2=sigma2, p2=pp2, a3=amp3, b3=cen3, c3=sigma3, p3=pp3, 
               error_a1=amp1_err, error_a2=amp2_err, error_a3=amp3_err, errordef=1., pedantic=False)
    m.migrad() # run optimiser
    ##chi square/dof
    chi_s = m.fval / (len(pulsex) - len(m.args)) 
    return np.array([m.values[0], m.values[1], m.values[2],m.values[3], 
                     m.values[4], m.values[5], m.values[6], m.values[7],
                     m.values[8], m.values[9], m.values[10], m.values[11],
                     m.errors[0], m.errors[1], m.errors[2], m.errors[3],
                     m.errors[4], m.errors[5], m.errors[6], m.errors[7],
                     m.errors[8], m.errors[9], m.errors[10], m.errors[11],
                     chi_s])

######Outputs from simple gaussian fit
def res1(amp, cen, amp_err):
    fit1 = get1(amp, cen, amp_err)
    a1 = fit1[0] #amplitude
    t1 = fit1[1] #center
    s1 = fit1[2] #sigma
    alp1 = fit1[3] #alpha 
    a1_e = fit1[4] #amplitude error
    t1_e = fit1[5] #center error
    s1_e = fit1[6] #sigma error
    ch1 = fit1[8] #chi square
    #####Q1    
    q1 = s1*a1
    q1_err = np.sqrt((s1_e/s1)**2+(a1_e/a1)**2)*q1
    
    if (a1>=0 and t1>=0):
        csvdata={'pixID':[pixelID],      
                 'row':[i],
                 'col':[j],
                 't1':["{:.3f}".format(t1)],
                 'dt1':["{:.3f}".format(t1_e)],
                 't2':['nan'],
                 'dt2':['nan'],
                 't3':['nan'],
                 'dt3':['nan'],
                 'dT1':['nan'],
                 'ddT1':['nan'],
                 'dT2':['nan'],
                 'ddT2':['nan'],
                 'q1':["{:.3f}".format(q1)],
                 'dq1':["{:.3f}".format(q1_err)],
                 'q2':['nan'],
                 'dq2':['nan'],
                 'q3':['nan'],
                 'dq3':['nan'],
                 'arcR':["{:.3f}".format(arcR)],
                 'elev':["{:.5f}".format(pixelelv)],
                 'azim':["{:.5f}".format(pixelazm2)],
                 'lat_pix':["{:.5f}".format(latpix)],
                 'lon_pix':["{:.5f}".format(lonpix)]} 
        arg=pd.DataFrame(csvdata)
        arg.to_csv(csvname,header=None, index=None, sep=',', mode='a')
        print('* Simple_elves', histname, gps, nanosec)
        #print('peaks = ',pulsex[peaks])
        print('Best fit: t1=',"{:.2f}".format(t1) ,'+/-', "{:.2f}".format(t1_e),'alpha=',"{:.2f}".format(alp1), 'chi1=',"{:.2f}".format(ch1))
        #print('eye=',eye, 'tel=', telescope, pixelelv, pixelazm2, 'col=',j,'row=', i)
        #plt.figure(figsize=(10,2))
        #plt.errorbar(pulsex, pulsey, errory, color='k', fmt='.',capsize=3)
        #plt.plot(pulsex, gauss(pulsex,a1,t1,s1,alp1), color='r', linewidth=4)
        #plt.grid(True)
        #plt.show()    

######Outputs from double gaussian fit
def res2(amp1, cen1, amp2, cen2, amp1_err, amp2_err):
    fit2 = get2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)
    a1 = fit2[0] #amplitude_1
    t1 = fit2[1] #center_1
    s1 = fit2[2] #sigma_1
    alp1 = fit2[3] #alpha_1
    a2 = fit2[4] #amplitude_2
    t2 = fit2[5] #center_2
    s2 = fit2[6] #sigma_2
    alp2 = fit2[7] #alpha_2
    a1_e = fit2[8] #amplitude error_1
    t1_e = fit2[9] #center error_1
    s1_e = fit2[10] #sigma error_1
    a2_e = fit2[12] #amplitude error_2
    t2_e = fit2[13] #center error_2
    s2_e = fit2[14] #sigma error_2
    ch2 = fit2[16] #chi square      
    
    if (a1>=0 and a2>=0 and t1>=0 and t2>=0):
        #####Delta T     
        delta_t1 = np.abs(t1 - t2)    
        dt_err1 = np.sqrt((t1_e)**2+(t2_e)**2)
        
        #####Q1    
        q1 = s1*a1
        q1_err = np.sqrt((s1_e/s1)**2+(a1_e/a1)**2)*q1
        
        #####Q2    
        q2 = s2*a2
        q2_err = np.sqrt((s2_e/s2)**2+(a2_e/a2)**2)*q2
        
        csvdata={'pixID':[pixelID],
                 'row':[i],
                 'col':[j],
                 't1':["{:.3f}".format(t1)],
                 'dt1':["{:.3f}".format(t1_e)],
                 't2':["{:.3f}".format(t2)],
                 'dt2':["{:.3f}".format(t2_e)],
                 't3':['nan'],
                 'dt3':['nan'],
                 'dT1':["{:.3f}".format(delta_t1)],
                 'ddT1':["{:.3f}".format(dt_err1)],
                 'dT2':['nan'],
                 'ddT2':['nan'],
                 'q1':["{:.3f}".format(q1)],
                 'dq1':["{:.3f}".format(q1_err)],
                 'q2':["{:.3f}".format(q2)],
                 'dq2':["{:.3f}".format(q2_err)],
                 'q3':['nan'],
                 'dq3':['nan'],
                 'arcR':["{:.3f}".format(arcR)],
                 'elev':["{:.5f}".format(pixelelv)],
                 'azim':["{:.5f}".format(pixelazm2)],
                 'lat_pix':["{:.5f}".format(latpix)],
                 'lon_pix':["{:.5f}".format(lonpix)]} 
        arg=pd.DataFrame(csvdata)
        arg.to_csv(csvname,header=None, index=None, sep=',', mode='a')
        
        print('* * Double_elves', histname, gps, nanosec)
        #print('peaks = ',pulsex[peaks])
        print('Best fit: t1=',"{:.2f}".format(t1) ,'+/-', "{:.2f}".format(s1),'alpha=',"{:.2f}".format(alp1),'t2=', "{:.2f}".format(t2) ,'+/-', "{:.2f}".format(s2),'alpha2=',"{:.2f}".format(alp2), 'chi2=',"{:.2f}".format(ch2))
        #print('dt1= ',delta_t1)
        #print('eye=',eye, 'tel=', telescope, pixelelv, pixelazm2, 'col=',j,'row=', i)
        #plt.figure(figsize=(10,2))
        #plt.errorbar(pulsex, pulsey, errory, color='k', fmt='.',capsize=3)
        #plt.plot(pulsex, gauss_2(pulsex,a1,t1,s1,alp1,a2,t2,s2,alp2), color='b', linewidth=4)
        #plt.grid(True)
        #plt.show()
        
def res3(amp1, cen1, amp2, cen2, amp3, cen3, amp1_err, amp2_err, amp3_err):
    fit3 = get3(amp1, cen1, amp2, cen2, amp3, cen3, amp1_err, amp2_err, amp3_err)
    a1 = fit3[0] #amplitude_1
    t1 = fit3[1] #center_1
    s1 = fit3[2] #sigma_1
    alp1 = fit3[3] #alpha_1
    a2 = fit3[4] #amplitude_2
    t2 = fit3[5] #center_2
    s2 = fit3[6] #sigma_2
    alp2 = fit3[7] #alpha_2
    a3 = fit3[8] #amplitude_3
    t3 = fit3[9] #center_3
    s3 = fit3[10] #sigma_3
    alp3 = fit3[11] #alpha_3
    a1_e = fit3[12] #amplitude error_1
    t1_e = fit3[13] #center error_1
    s1_e = fit3[14] #sigma error_1
    a2_e = fit3[16] #amplitude error_2
    t2_e = fit3[17] #center error_2
    s2_e = fit3[18] #sigma error_2
    a3_e = fit3[20] #amplitude error_3
    t3_e = fit3[21] #center error_3
    s3_e = fit3[22] #sigma error_3
    ch3 = fit3[24] #chi square      
    
    if (a1>=0 and a2>=0 and a3>=0 and t1>=0 and t2>=0 and t3>=0):
        #####Delta T1 
        delta_t1 = np.abs(t1 - t2)
        dt_err1 = np.sqrt((t1_e)**2+(t2_e)**2)
        
        #####Delta T2 
        delta_t2 = np.abs(t1 - t3)
        dt_err2 = np.sqrt((t1_e)**2+(t3_e)**2)
        
        #####Q1    
        q1 = s1*a1
        q1_err = np.sqrt((s1_e/s1)**2+(a1_e/a1)**2)*q1
        
        #####Q2    
        q2 = s2*a2
        q2_err = np.sqrt((s2_e/s2)**2+(a2_e/a2)**2)*q2
        
        #####Q3    
        q3 = s3*a3
        q3_err = np.sqrt((s3_e/s3)**2+(a3_e/a3)**2)*q3
        
        csvdata={'pixID':[pixelID],             
                 'row':[i],
                 'col':[j],
                 't1':["{:.3f}".format(t1)],
                 'dt1':["{:.3f}".format(t1_e)],
                 't2':["{:.3f}".format(t2)],
                 'dt2':["{:.3f}".format(t2_e)],
                 't3':["{:.3f}".format(t3)],
                 'dt3':["{:.3f}".format(t3_e)],
                 'dT1':["{:.3f}".format(delta_t1)],
                 'ddT1':["{:.3f}".format(dt_err1)],
                 'dT2':["{:.3f}".format(delta_t2)],
                 'ddT2':["{:.3f}".format(dt_err2)],
                 'q1':["{:.3f}".format(q1)],
                 'dq1':["{:.3f}".format(q1_err)],
                 'q2':["{:.3f}".format(q2)],
                 'dq2':["{:.3f}".format(q2_err)],
                 'q3':["{:.3f}".format(q3)],
                 'dq3':["{:.3f}".format(q3_err)],
                 'arcR':["{:.3f}".format(arcR)],
                 'elev':["{:.5f}".format(pixelelv)],
                 'azim':["{:.5f}".format(pixelazm2)],
                 'lat_pix':["{:.5f}".format(latpix)],
                 'lon_pix':["{:.5f}".format(lonpix)]} 
        arg=pd.DataFrame(csvdata)
        arg.to_csv(csvname,header=None, index=None, sep=',', mode='a')
        
        print('* * * Triple_elves', histname, gps, nanosec)
        #print('peaks = ',pulsex[peaks])
        print('Best fit: t1=',"{:.2f}".format(t1) ,'+/-', "{:.2f}".format(s1),'alpha=',"{:.2f}".format(alp1),'t2=', "{:.2f}".format(t2) ,'+/-', "{:.2f}".format(s2),'alpha2=',"{:.2f}".format(alp2),'t3=', "{:.2f}".format(t3) ,'+/-', "{:.2f}".format(s3),'alpha2=',"{:.2f}".format(alp3), 'chi3=',"{:.2f}".format(ch3))
        #print('dt1= ',delta_t1,'dt2= ',delta_t2 )
        #print('eye=',eye, 'tel=', telescope, pixelelv, pixelazm2, 'col=',j,'row=', i)
        #plt.figure(figsize=(10,2))
        #plt.errorbar(pulsex, pulsey, errory, color='k', fmt='.',capsize=3)
        #plt.plot(pulsex, gauss_3(pulsex,a1,t1,s1,alp1,a2,t2,s2,alp2,a3,t3,s3,alp3), color='g', linewidth=4)
        #plt.grid(True)
        #plt.show()
        

############################################################
#####Choose one option for running the code for each storm 
#event = np.loadtxt('input_files/november2018.txt')
#event = np.loadtxt('input_files/december2019.txt')
#event = np.loadtxt('input_files/april2020.txt')
event = np.loadtxt('input_files/march2021.txt')
#############################################################

sites = pd.read_csv('auxiliary/latlonsite.csv') #location of the sites
pixelMap = pd.read_csv('auxiliary/fdPixelMap_5sites.csv') #pixel elevation and azimuth
hsites = sites.hsite #km
latsites = sites.latsite #degrees
lonsites = sites.lonsite #deg
backwall = sites.backwall #deg
pixelel = pixelMap.elevation #deg
pixelazi = pixelMap.azimuth #deg



for ev in range(len(event)):
#for ev in range(1,2):
    #Data from the events 
    gps=int(event[ev,0])
    nanosec=int(event[ev,1])
    bay_1 =int(event[ev,2])
    bay_2 =int(event[ev,3])
    bays = np.array([bay_1,bay_2])
    eye = int(event[ev,4])
    lat_bolt = event[ev,5]
    lon_bolt = event[ev,6]
    t_0 = event[ev,7] 
    #Peak finder parameters
    hg = event[ev,8] 
    wd = event[ev,9]
    pr = event[ev,10]
    #Peak overlap factor 
    pof = event[ev,11] # pof=1 for very separated peaks, and pof=0.5 for overlapping peaks
    #print(bays)
    
    #FD pixels data  
    lat_site = latsites[eye-1] #degrees from (0-4) since eyes are (1-5)
    lon_site = lonsites[eye-1] #degrees
    bw_site = math.radians(backwall[eye-1])
    Hsite = hsites[eye-1] #km
    Hiono = 92.0 #km 
    Rearth = 6371.0 #km
    Rsite = Rearth + Hsite 
    Riono = Rearth + Hiono
    #Site vector components
    phi = math.radians(lon_site) 
    gamma = math.radians(90-lat_site)
    ax = Rsite*math.sin(gamma)*math.cos(phi)
    ay = Rsite*math.sin(gamma)*math.sin(phi)
    az = Rsite*math.cos(gamma)
    
    #Importing root files and setting the csv file name
    #Root files path:
    ############################################################
    #####Choose one option for running the code for each storm 
    #rootname = 'input_files/root_files/november2018/EVT%d'%(gps)+'-%d'%(nanosec)+'.root' 
    #csvname='output_files/november2018/EVT%d'%(gps)+'-%d'%(nanosec)+'.csv'
    #rootname = 'input_files/root_files/december2019/EVT%d'%(gps)+'-%d'%(nanosec)+'.root' 
    #csvname='output_files/december2019/EVT%d'%(gps)+'-%d'%(nanosec)+'.csv'
    #rootname = 'input_files/root_files/april2020/EVT%d'%(gps)+'-%d'%(nanosec)+'.root' 
    #csvname='output_files/april2020/EVT%d'%(gps)+'-%d'%(nanosec)+'.csv'
    #rootname = 'input_files/root_files/march2021/EVT%d'%(gps)+'-%d'%(nanosec)+'.root' 
    #csvname='output_files/march2021/EVT%d'%(gps)+'-%d'%(nanosec)+'2.csv'
    rootname = 'examples/root_files/EVT%d'%(gps)+'-%d'%(nanosec)+'.root'
    csvname = 'examples/output_files/EVT%d'%(gps)+'-%d'%(nanosec)+'.csv'
    #############################################################

    print('********************************************************')
    print('********************************************************')
    print('*******Reading', rootname)
    print('********************************************************')
    print('********************************************************')

    for m in range(len(bays)):
        if bays[m]>0:
            telescope=bays[m]
            #####Importing signal from each pixel
            for j in range(1,21):#from 1 to 21 for 20 cols
                for i in range(1,23):#from 1 to 23 for 22 rows
                    #####Number of pixel
                    pixel = i-1 +(j-1)*22
                    pix_array = []
                    pix_array.append(pixel)
                    
                    #####ArcR Distance calculation 
                    pixelID = int(pixel + (telescope-1)*440) #for setting the pixel elevation and azimuth 
                    #This pixelID is the number of the line in the fdPixelMap_5sites.csv, is not the first colum of the file, so pixelID range is (0,13199)
                    pixelelv = pixelel[pixelID] #deg
                    pixelazm2= pixelazi[pixelID] #deg
                    pixelazm = math.radians(pixelazi[pixelID]) #radians
                    beta = math.radians(90 - pixelelv) #radians  
                    #print(j, i , pixel, pixelID, telescope, pixelelv, pixelazm, pixelMap[pixelID:pixelID+1] )
                    ######Calculating r
                    r = var("r")
                    x1,x2 = solve(r**2 + 2*Rsite*math.cos(beta)*r + Rsite**2-Riono**2)
                    if x1 > x2:
                        r=x1
                    else:
                        r=x2
                    ######Calculating alpha
                    alpha = math.acos((Rsite**2+Riono**2-r**2)/(2*Rsite*Riono))
                    ######Calculating rot axis vector (erot)
                    poloN = (0.,0.,1.)
                    a = (ax,ay,az)
                    east1 = np.cross(poloN,a)
                    east = rotve(east1,a,pixelazm+bw_site)
                    erot = np.cross(a,east)
                    ######Calculating C
                    C = rotve(a,erot,alpha)
                    Cx = C[0]
                    Cy = C[1]
                    Cz = C[2]
                    ######Calculate lat, lon pixel C
                    radius = math.sqrt(Cx**2 + Cy**2 + Cz**2)
                    polar = math.acos(Cz/radius)
                    azimuthal = math.atan2(Cy, Cx)
                    latpix = 90-math.degrees(polar)
                    lonpix = math.degrees(azimuthal)
                    ######Calculating oblate distance (ArcR)
                    pixelLocation = latpix, lonpix
                    boltLocation = lat_bolt, lon_bolt
                    distance = lx.oblate_distance(boltLocation, pixelLocation) 
                    arcR = distance/1000 #km
                    
                    #####Import histograms 
                    file = uproot.open(rootname)
                    histname = "QCSvsTime%d"%(m)+"_R%d"%(i)+"_C%d"%(j)
                    xval = file[histname].axis().edges() #x values from the histogram
                    xval = xval[:-1]
                    yval = file[histname].values() #y values from the histogram
                    y_err = file[histname].errors() #y error values from the histogram
                    
                    pulse = []    
                    for k in range(len(xval)):
                        npulse = [xval[k], yval[k], y_err[k]]
                        pulse.append(npulse)
                    
                    arraytest = np.array(pulse)
                    pulse_c = arraytest[np.all(arraytest != 0.0, axis=1)] #to get non zero values                              
                    #####If using QCSvsTime
                    pulsex = pulse_c[:,0]*2 #x values in micro seconds
                    pulsey = pulse_c[:,1] #y values in photons
                    errory = pulse_c[:,2] #y error values in photons
                    #plt.errorbar(pulsex, pulsey, errory, color='k', fmt='.',capsize=3,label=histname)
                    #plt.legend()
                    #plt.show()  
                        
                    #####Looking for the main peaks of the signal
                    p=None
                    peaks=[]
                    for n in errory: 
                        if n != 0.: #because some signals are empty and p=max(errory) gets an error
                            p = pr*max(errory)
                            peaks, properties = find_peaks(pulsey, height=hg, prominence=p, width=wd) 
                            break
                        
                    #####Setting gaussian parameters based on the peaks found          
                    if len(peaks) == 1: 
                        #Inputs from peaks found 
                        amp1 = pulsey[peaks[0]] #amplitude
                        cen1 = pulsex[peaks[0]] #center
                        amp_err1 = errory[peaks[0]] #amplitude error
                        #print('SIMPLE SIMPLE')
                        res1(amp1, cen1, amp_err1) #Results from simple gaussian fit
                        
                    if len(peaks) == 2: 
                        amp1 = pulsey[peaks[0]] #first gaussian amplitude
                        cen1 = pulsex[peaks[0]] #first gaussian center
                        amp1_err = errory[peaks[0]] #first gaussian amplitude error
                        amp2 = pulsey[peaks[1]] #second gaussian amplitude
                        cen2 = pulsex[peaks[1]] #second gaussian center
                        amp2_err = errory[peaks[1]] #second gaussian amplitude error
                        ch1 = get1(amp1, cen1, amp1_err)[8] #chi square from simple gaussian fit
                        ch2 = get2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)[16] #chi square from double gaussian fit 
                        if ch1<=ch2:
                            #Keeping the simple gaussian fit 
                            #print('SIMPLE FROM DOUBLE')
                            res1(amp1, cen1, amp_err1)
                        else:
                            fit2 = get2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)
                            t1 = fit2[1]#center1
                            s1 = pof*fit2[2]#sigma1
                            t2 = fit2[5]#center2
                            s2 = pof*fit2[6]#sigma2
                                
                            if((t2-s2)>(t1+s1)): #checking if t2 and t1 are separable 
                                #print('DOUBLE')
                                res2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)                        
                            else:
                                #Keeping the single gaussian fit
                                #print('SIMPLE FROM DOUBLE with intersection')
                                res1(amp1, cen1, amp1_err)
                        
                    if len(peaks) == 3: 
                        amp1 = pulsey[peaks[0]] #first gaussian amplitude
                        cen1 = pulsex[peaks[0]] #first gaussian center
                        amp1_err = errory[peaks[0]] #first gaussian amplitude error
                        amp2 = pulsey[peaks[1]]#second gaussian amplitude
                        cen2 = pulsex[peaks[1]] #second gaussian center
                        amp2_err = errory[peaks[1]] #second gaussian amplitude error
                        amp3 = pulsey[peaks[2]] #third gaussian amplitude
                        cen3 = pulsex[peaks[2]] #third gaussian center
                        amp3_err = errory[peaks[2]]#third gaussian amplitude error
                        ch1 = get1(amp1, cen1, amp1_err)[8] #chi square from simple gaussian fit
                        ch2_1 = get2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)[16] #chi square from double gaussian fit t1 and t2
                        ch2_2 = get2(amp1, cen1, amp3, cen3, amp1_err, amp3_err)[16] #chi square from double gaussian fit t1 and t3
                        ch2_3 = get2(amp2, cen2, amp3, cen3, amp2_err, amp3_err)[16] #chi square from double gaussian fit t2 and t3
                        ch3 = get3(amp1, cen1, amp2, cen2, amp3, cen3, amp1_err, amp2_err, amp3_err)[24] #chi square from triple gaussian fit 
                        ch =  np.array([ch1, ch2_1, ch2_2, ch2_3, ch3])
                            
                        if (ch.min()==ch1):
                            #Keeping the simple gaussian fit 
                            #print('****SIMPLE FROM TRIPLE')
                            res1(amp1, cen1, amp1_err)      
                            
                        if (ch.min()==ch2_1): #fit with t1 and t2 
                            #Keeping the double gaussian fit 
                            fit2 = get2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)
                            t1 = fit2[1]#center1
                            s1 = pof*fit2[2]#sigma1
                            t2 = fit2[5]#center2
                            s2 = pof*fit2[6]#sigma2
                                
                            if((t2-s2)>(t1+s1)): #checking if t2 and t1 are separable 
                                #print('****DOUBLE FROM TRIPLE')
                                res2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)
                            else:
                                #Keeping the single gaussian fit
                                #print('****SIMPLE FROM TRIPLE with intersection')
                                res1(amp1, cen1, amp1_err)
                                    
                        if (ch.min()==ch2_2): #fit with t1 and t3 
                            #Keeping the double gaussian fit 
                            fit2 = get2(amp1, cen1, amp3, cen3, amp1_err, amp3_err)
                            t1 = fit2[1]#center1
                            s1 = pof*fit2[2]#sigma1
                            t2 = fit2[5]#center2
                            s2 = pof*fit2[6]#sigma2
                                
                            if((t2-s2)>(t1+s1)): #checking if t2 and t1 are separable 
                                #print('****DOUBLE FROM TRIPLE')
                                res2(amp1, cen1, amp3, cen3, amp1_err, amp3_err)
                            else:
                                #Keeping the single gaussian fit
                                #print('****SIMPLE FROM TRIPLE with intersection')
                                res1(amp1, cen1, amp1_err)  
                                        
                        if (ch.min()==ch2_3): #fit with t2 and t3 
                            #Keeping the double gaussian fit 
                            fit2 = get2(amp2, cen2, amp3, cen3, amp2_err, amp3_err)
                            t1 = fit2[1]#center1
                            s1 = pof*fit2[2]#sigma1
                            t2 = fit2[5]#center2
                            s2 = pof*fit2[6]#sigma2
                                
                            if((t2-s2)>(t1+s1)): #checking if t2 and t1 are separable 
                                #print('****DOUBLE FROM TRIPLE')
                                res2(amp2, cen2, amp3, cen3, amp2_err, amp3_err)
                            else:
                                #Keeping the single gaussian fit
                                #print('****SIMPLE FROM TRIPLE with intersection')
                                res1(amp1, cen1, amp1_err)                              
                                    
                        if (ch.min()==ch3): 
                            fit3 = get3(amp1, cen1, amp2, cen2, amp3, cen3, amp1_err, amp2_err, amp3_err)
                            t1 = fit3[1]#center1
                            s1 = pof*fit3[2]#sigma1
                            t2 = fit3[5]#center2
                            s2 = pof*fit3[6]#sigma2
                            t3 = fit3[9]#center2
                            s3 = pof*fit3[10]#sigma2
                                
                            if(((t2-s2)>(t1+s1))and((t3-s3)>(t2+s2))):
                                #print('****TRIPLE')
                                res3(amp1, cen1, amp2, cen2, amp3, cen3, amp1_err, amp2_err, amp3_err)  
                                
                            elif((t2-s2)<(t1+s1)): # t1 and t2 intersection, fit with t1 and t3
                                #print('****DOUBLE FROM T1 AND T2 intersection')
                                res2(amp1, cen1, amp3, cen3, amp1_err, amp3_err)  
                                
                            elif((t3-s3)<(t2+s2)): # t2 and t3 intersection, fit with t1 and t2
                                #print('****DOUBLE FROM T2 AND T3 intersection')
                                res2(amp1, cen1, amp2, cen2, amp1_err, amp2_err)
                                
                            else:
                                #Keeping the single gaussian fit
                                #print('****SIMPLE FROM TRIPLE with intersection')
                                res1(amp1, cen1, amp1_err)
                    
                    
                
            

FileNotFoundError: input_files/march2021.txt not found.